# Suite EMB — Value objects, union values and intersection values

A property whose schema is an `OfObject` holds a value object: a read-only record with no identity, written nested in
its owner's snapshot. Union and intersection values are records too, whose properties are the union's branches or the
intersection's parts, by name. A union value holds exactly one branch (`{"phone": {...}}` in snapshots), and writing a
branch clears any other; an intersection value holds each of its parts (`{"stamp": {...}, "audit": {...}}`).

In [ ]:
import copy

from mbse.Schemas.Framework import Comparison as C, JSON, Modules, Plain, Proxies, Reachable, Schemas as S, Validators, YAML
from mbse.Schemas.Framework.Errors import DecodeError
from support import Fake, raises, set_property, text

store = Proxies.OfStore()

Phone = S.OfObject.Builder().properties(text("number"), text("label")).create()
Email = S.OfObject.Builder().properties(text("address")).create()
Reach = S.OfUnion.Builder().branches(lambda b: b.name("phone").of(Phone), lambda b: b.name("email").of(Email)).create()
Id = S.OfUnion.Builder().branches(lambda b: b.name("number").of(lambda t: t.as_native(int)),
                                  lambda b: b.name("code").of(lambda t: t.as_native(str))).create()
Contact = S.OfObject.Builder().name("Contact").ref().properties(
    text("name"), lambda p: p.name("home").of(Phone), lambda p: p.name("reach").of(Reach),
    lambda p: p.name("id").of(Id)).create()
store.register(Contact)
B = store


class Written:
    """A `Visitable` whose `accept` runs `write(visitor)`: data that neither proxies nor fakes produce."""

    def __init__(self, schema_name, write):
        self._schema_name, self._write = schema_name, write

    def identity(self):
        return id(self)

    def schema_name(self):
        return self._schema_name

    def owner(self):
        return None

    def accept(self, visitor):
        self._write(visitor)


def union(name, write):
    """A `Written` Contact whose property `name` is written as a union value by `write(union visitor)`."""
    return Written("Contact", lambda v: v.property(name, lambda p: p.value(lambda a: a.as_union(write))))

## EMB-01 · A value object's schema may have adjacencies, however it is held

In [ ]:
assert Contact.validate() == [] and Reach.validate() == [] and Id.validate() == []
Owner = S.OfObject.Builder().properties(text("x")).create()
Links = S.OfRelation.Builder().links("a", "b").create()
S.OfObject.Builder(Owner).relations(lambda r: r.name("links").of(Links).me("a")).update()
Either = S.OfUnion.Builder().branches(lambda b: b.name("owner").of(Owner), lambda b: b.name("phone").of(Phone)).create()
Both = S.OfIntersection.Builder().parts(lambda p: p.name("owner").of(Owner), lambda p: p.name("phone").of(Phone)).create()
Holder = S.OfObject.Builder().properties(lambda p: p.name("owner").of(Owner), lambda p: p.name("either").of(Either),
                                         lambda p: p.name("both").of(Both)).create()
assert Holder.validate() == []  # directly, as a union's branch and as an intersection's part
Pair = S.OfRelation.Builder().links("a", "b").properties(lambda p: p.name("inner").of(Owner)).create()
assert Pair.validate() == ["property 'inner': a value object held by an entry cannot have adjacencies"]

## EMB-02 · Value objects: set with a Spec or a record, read, updated in part, cleared

In [ ]:
alice = B.Contact().name("Alice").home(lambda r: r.number("+44 1").label("home")).create()
assert alice.home.number == "+44 1" and alice.home.label == "home"
assert isinstance(alice.home, Proxies.OfObject.Record)
assert B.Contact().home(lambda r: r.number("1")).create().home.label is None
with raises(AttributeError, match="value objects are read-only; use a builder"):
    alice.home.number = "x"
with raises(AttributeError, match="nope"):
    alice.home.nope
assert B.Contact(alice).home(lambda r: r.label("work")).update() is alice  # starts from the value already set
assert alice.home.number == "+44 1" and alice.home.label == "work"
bob = B.Contact().name("Bob").home(alice.home).create()  # a record is a value, copied into the new owner
assert bob.home.number == "+44 1" and bob.home.label == "work"
B.Contact(bob).clear("home").update()
assert bob.home is None
assert alice.home.label == "work"
with raises(TypeError, match="property 'home' takes a value object or a Spec, got str"):
    B.Contact().home("+44")
with raises(AttributeError, match="'nope' is not a property of the value object"):
    B.Contact().home(lambda r: r.nope(1))

## EMB-03 · Union values: one branch, set and read by name; writing a branch clears any other

In [ ]:
carol = B.Contact().reach(lambda u: u.email(lambda r: r.address("c@x"))).id(lambda u: u.number(7)).create()
assert carol.reach.email.address == "c@x" and carol.id.number == 7
assert isinstance(carol.reach, Proxies.OfObject.Record)
assert carol.reach.phone is None
with raises(AttributeError, match="union values are read-only; use a builder"):
    carol.id.number = 8
dan = B.Contact().reach(carol.reach).id(lambda u: u.code("D-1")).create()  # a record is a value
assert dan.reach.email.address == "c@x" and dan.id.code == "D-1"
B.Contact(dan).id(lambda u: u.number(8)).update()
assert dan.id.number == 8
assert dan.id.code is None  # the other branch was cleared
B.Contact(dan).reach(lambda u: u.email(lambda r: r)).update()
assert dan.reach.email.address == "c@x"  # starts from the value already set
with raises(TypeError, match="property 'id' takes a union value or a Spec, got int"):
    B.Contact().id(7)
with raises(AttributeError, match="'fax' is not a branch of the union"):
    B.Contact().reach(lambda u: u.fax(1))
with raises(AttributeError, match="'fax' is not a branch of the union"):
    B.Contact().reach(lambda u: u.property("fax", lambda p: None))
with raises(AttributeError, match="a union value has no adjacencies, got 'x'"):
    B.Contact().reach(lambda u: u.adjacency("x", lambda a: None))
seen = []
B.Contact(carol).property("reach", lambda p: p.value(lambda a: a.as_union(lambda u: (
    u.properties(lambda q: seen.append(q.name())), seen.append(u.has("phone"))))))
assert seen == ["email", False]
B.Contact(carol).id(lambda u: u.clear("number")).update()  # a union value without a branch is no value
assert carol.id is None
with raises(TypeError, match="property 'name' does not hold an object"):
    B.Contact().property("name", lambda p: p.value(lambda a: a.as_object(lambda o: None)))
with raises(TypeError, match="property 'name' does not hold a union"):
    B.Contact().property("name", lambda p: p.value(lambda a: a.as_union(lambda u: None)))

## EMB-04 · Snapshots: value objects and union values nested, a union value by its branch; strict decoding

In [ ]:
eve = (B.Contact().name("Eve").home(lambda r: r.number("1"))
       .reach(lambda u: u.phone(lambda r: r.number("2").label("m"))).id(lambda u: u.code("E")).create())
expected = JSON.ToJSON(store)(Contact, eve)
assert expected == ('{"root": "s0", "objects": {"s0": {"name": "Eve", "home": {"number": "1"}, "reach": {"phone": '
                    '{"number": "2", "label": "m"}}, "id": {"code": "E"}}}}')
for encode, decode in [(Plain.ToPlain(store), Plain.FromPlain), (JSON.ToJSON(store), JSON.FromJSON), (YAML.ToYAML(store), YAML.FromYAML)]:
    assert JSON.ToJSON(store)(Contact, decode(B)(Contact, encode(Contact, eve))) == expected

FromPlain = Plain.FromPlain(B)
for s0, message in [
    ({"home": 3}, "$.objects.s0.home: a value object must be a mapping, got int"),
    ({"home": {"x": "1"}}, "$.objects.s0.home.x: the value object has no property 'x'"),
    ({"home": {"number": 1}}, "$.objects.s0.home.number: expected str, got int"),
    ({"reach": "1"}, "$.objects.s0.reach: a union value must be a mapping, got str"),
    ({"reach": {"number": "1"}}, "$.objects.s0.reach.number: the union has no branch 'number'"),
    ({"id": {"$branch": 0, "$value": 1}}, "$.objects.s0.id.$branch: the union has no branch '$branch'"),
    ({"id": {}}, "$.objects.s0.id: a union value holds exactly one branch, got 0"),
    ({"id": {"number": 1, "code": "x"}}, "$.objects.s0.id: a union value holds exactly one branch, got 2"),
    ({"id": {"number": "x"}}, "$.objects.s0.id.number: expected int, got str"),
]:
    with raises(DecodeError, match=message):
        FromPlain(Contact, {"root": "s0", "objects": {"s0": s0}})
with raises(TypeError, match="property 'home' is not native"):
    Plain.ToPlain(store)(Contact, Fake("Contact", {"home": "x"}))
# A writer given two branches keeps the last, and a union value written without a branch is left out.
assert Plain.ToPlain(store)(Contact, union("id", lambda u: (set_property(u, "number", 1), set_property(u, "code", "c")))) == {
    "root": "s0", "objects": {"s0": {"id": {"code": "c"}}}}
assert Plain.ToPlain(store)(Contact, union("id", lambda u: None)) == {"root": "s0", "objects": {"s0": {}}}

Box = S.OfObject.Builder().name("Box").ref().properties(text("x")).create()
Slot = S.OfRelation.Builder().name("Slot").links("box", "item").properties(lambda p: p.name("tag").of(Phone)).create()
S.OfObject.Builder(Box).relations(lambda r: r.name("items").of(Slot).me("box"),
                                  lambda r: r.name("boxes").of(Slot).me("item")).update()
store.register(Box)
store.register(Slot)
with raises(DecodeError, match="$.objects.s0.items[0].tag: a value object must be a mapping, got str"):
    FromPlain(Box, {"root": "s0", "objects": {"s0": {"items": [{"item": {"$ref": "s0", "$schema": "Box"}, "tag": "x"}]}}})

## EMB-05 · Validation: embedded values recursively, and a union value's one branch

In [ ]:
validate = Validators.Validate(B)
assert validate(Contact, eve) == []
assert validate(Contact, Fake("Contact", {"home": "x", "reach": 3})) == [
    "Contact#0.home: expected a value object, got str", "Contact#0.reach: expected a union value, got int"]
assert validate(Contact, B.Contact().home(lambda r: r.number(5)).create()) == ["Contact#0.home.number: expected str, got int"]
stray = Written("Contact", lambda v: v.property("home", lambda p: p.value(lambda a: a.as_object(
    lambda o: set_property(o, "fax", "1")))))
assert validate(Contact, stray) == ["Contact#0.home.fax: not a property of the value object"]
assert validate(Contact, union("id", lambda u: set_property(u, "fax", 1))) == [
    "Contact#0.id.fax: not a branch of the union"]
assert validate(Contact, union("id", lambda u: (set_property(u, "number", 1), set_property(u, "code", "c")))) == [
    "Contact#0.id: a union value holds exactly one branch, got 2"]  # the recorder keeps both, to report them
assert validate(Contact, union("id", lambda u: None)) == [
    "Contact#0.id: a union value holds exactly one branch, got 0"]
assert validate(Contact, union("id", lambda u: set_property(u, "number", "1"))) == [
    "Contact#0.id.number: expected int, got str"]
assert validate(Contact, Written("Contact", lambda v: v.property("id", lambda p: p.value(lambda a: a.as_object(
    lambda o: None))))) == ["Contact#0.id: expected a union value, got a value object"]
assert validate(Contact, union("name", lambda u: None)) == ["Contact#0.name: expected str, got a union value"]
Means = S.OfIntersection.Builder().parts(lambda p: p.name("phone").of(Phone), lambda p: p.name("email").of(Email)).create()
Mixed = S.OfObject.Builder().name("Mixed").ref().properties(lambda p: p.name("both").of(Means)).create()
store.register(Mixed)
assert validate(Mixed, Fake("Mixed", {"both": 1})) == ["Mixed#0.both: expected an intersection value, got int"]
with raises(DecodeError, match="$.objects.s0.both: an intersection value must be a mapping, got int"):
    FromPlain(Mixed, {"root": "s0", "objects": {"s0": {"both": 1}}})
box = Fake("Box")
box.adjacencies["items"] = [{"item": box, "tag": "x"}]
assert validate(Box, box) == ["Box#0.items[0].tag: expected a value object, got str"]

values = Validators.properties_of(eve)
assert Validators.properties_of(values["id"]) == {"code": "E"}  # a union value is a record of its branch
assert Validators.properties_of(values["home"]) == {"number": "1"}
assert Validators.properties_of(Validators.properties_of(values["reach"])["phone"]) == {"number": "2", "label": "m"}
Tagged = S.OfObject.Builder().properties(lambda p: p.name("id").of(Id)).create()
Sleeve = S.OfObject.Builder().name("Sleeve").ref().properties(lambda p: p.name("tagged").of(Tagged)).create()
store.register(Sleeve)
sleeve = B.Sleeve().tagged(lambda r: r.id(lambda u: u.number(5))).create()
tagged = Validators.properties_of(Validators.properties_of(sleeve)["tagged"])  # replays the union value it holds
assert Validators.properties_of(tagged["id"]) == {"number": 5}

## EMB-06 · Comparison: value objects by their properties, union values by branch and value

In [ ]:
def record(value):
    return C.OfObject(Contact, value)


assert record(eve).compare(record(B.Contact(eve).clone())) == 0
assert record(eve).compare(record(B.Contact(eve).home(lambda r: r.label("x")).clone())) is None
assert record(eve).compare(record(B.Contact(eve).id(lambda u: u.number(3)).clone())) is None  # another branch
assert record(eve).compare(record(B.Contact(eve).id(lambda u: u.code("F")).clone())) is None
assert record(B.Contact().create()).compare(record(B.Contact().create())) == 0
number = C.OfUnion(Id)
assert number.compare(C.OfUnion(Id)) == 0  # both hold no branch
with raises(KeyError, match="unknown property 'fax'"):
    number.property("fax", lambda p: None)
set_property(number, "number", 1)
assert number.has("number") and not number.has("code") and number.compare(C.OfUnion(Id)) is None
other = C.OfUnion(Id)
set_property(other, "number", 1)
assert number.compare(other) == 0 and number.compare(C.OfUnion(Reach)) is None  # another union
set_property(other, "code", "a")  # writing a branch clears any other
seen = []
assert other.properties(lambda p: seen.append(p.name())) is other and seen == ["code"]
assert number.compare(other) is None and other.clear("code").compare(C.OfUnion(Id)) == 0
word = C.OfAny(S.OfNative.Data(str)).as_native(lambda n: n.set("b"))
assert C.OfAny(Id).as_union(lambda u: set_property(u, "code", "b")).compare(word) is None
assert C.OfAny(Id).as_union(lambda u: None).compare(C.OfAny(Id)) == 0  # a union value without a branch is absent
twice = C.OfAny(Id)
twice.as_union(lambda u: set_property(u, "code", "a"))
twice.as_union(lambda u: set_property(u, "code", "b"))  # writing again reuses the recorded value
assert twice.compare(C.OfAny(Id).as_union(lambda u: set_property(u, "code", "b"))) == 0

## EMB-07 · The new visitor classes read back through their protocols

In [ ]:
# Proxies: a value object's builder, read through Visitors.OfObject.
seen = []
B.Contact(alice).property("home", lambda p: p.value(lambda a: a.as_object(lambda r: (
    r.properties(lambda q: seen.append(q.name())), seen.append(r.has("label")), r.clear("label"),
    seen.append(r.has("label")), r.adjacencies(seen.append)))))
assert seen == ["number", "label", True, False]
with raises(AttributeError, match="a value object has no adjacencies, got 'x'"):
    B.Contact().home(lambda r: r.adjacency("x", lambda a: None))
with raises(AttributeError, match="'nope' is not a property of the value object"):
    B.Contact().home(lambda r: r.property("nope", lambda q: None))
B.Contact().home(lambda r: seen.append(hasattr(r, "_private")))  # Python's own probes are not properties
assert seen[-1] is False

# Plain: writers for value objects and union values, and their errors.
out = {}
writer = Plain._ObjectWriter(out, Contact, lambda t: {})
writer.property("home", lambda p: p.value(lambda a: a.as_object(lambda o: set_property(o, "number", "1"))))
writer.property("home", lambda p: p.value(lambda a: a.as_object(lambda o: set_property(o, "label", "h"))))
assert out == {"home": {"number": "1", "label": "h"}}  # writing again adds to the same mapping
with raises(TypeError, match="property 'name' is not an object"):
    writer.property("name", lambda p: p.value(lambda a: a.as_object(lambda o: None)))
with raises(TypeError, match="property 'name' is not a union"):
    writer.property("name", lambda p: p.value(lambda a: a.as_union(lambda u: None)))
writer.property("id", lambda p: p.value(lambda a: a.as_union(lambda u: (
    set_property(u, "number", 1), seen.append(u.has("number")), set_property(u, "code", "I"),
    seen.append(u.has("number"))))))
assert out["id"] == {"code": "I"} and seen[-2:] == [True, False]
with raises(KeyError, match="unknown property 'fax'"):
    writer.property("id", lambda p: p.value(lambda a: a.as_union(lambda u: set_property(u, "fax", 1))))
assert out["id"] == {"code": "I"}  # an unknown branch clears nothing
writer.property("home", lambda p: p.value(lambda a: a.as_object(lambda o: o.identify(alice.home))))
assert "$id" not in out["home"]  # outside a snapshot, nothing links a value object

# Validators: the recorder for union values.
recorder = Validators._ObjectRecord()
recorder.property("id", lambda p: p.value(lambda a: a.as_union(lambda u: set_property(u, "number", 1))))
recorder.property("id", lambda p: p.value(lambda a: a.as_union(lambda u: seen.append(u.has("number")))))
recorder.property("home", lambda p: p.value(lambda a: a.as_object(lambda o: set_property(o, "number", "1"))))
recorder.property("home", lambda p: p.value(lambda a: a.as_object(lambda o: set_property(o, "label", "h"))))
assert Validators.properties_of(Validators.properties_of(recorder)["home"]) == {"number": "1", "label": "h"}
assert seen[-1] is True  # writing again reuses the recorded union value
recorder.property("id", lambda p: p.value(lambda a: a.as_object(lambda o: None)))  # another kind replaces it
assert Validators._kind(Validators.properties_of(recorder)["id"]) == "a value object"

## EMB-08 · Intersection values: each part by name; parts do not merge

In [ ]:
Stamp = S.OfObject.Builder().properties(text("updated")).create()
Audit = S.OfObject.Builder().properties(text("by"), lambda p: p.name("updated").of(lambda t: t.as_native(int))).create()
Meta = S.OfIntersection.Builder().parts(lambda p: p.name("stamp").of(Stamp), lambda p: p.name("audit").of(Audit)).create()
Code = S.OfIntersection.Builder().parts(lambda p: p.name("low").of(lambda t: t.as_native(int)),
                                        lambda p: p.name("high").of(lambda t: t.as_native(int))).create()
Doc = S.OfObject.Builder().name("Doc").ref().properties(text("title"), lambda p: p.name("meta").of(Meta),
                                      lambda p: p.name("code").of(Code)).create()
store.register(Doc)
assert Doc.validate() == [] and Meta.validate() == []  # 'updated' is a str in one part and an int in the other

# Proxies: each part set and read by name.
doc = (B.Doc().title("T").meta(lambda i: i.stamp(lambda r: r.updated("2026")).audit(lambda r: r.by("ann").updated(1)))
       .code(lambda i: i.low(1).high(9)).create())
assert doc.meta.audit.by == "ann" and doc.meta.stamp.updated == "2026" and doc.meta.audit.updated == 1
assert doc.code.low == 1 and doc.code.high == 9
assert B.Doc(doc).meta(lambda i: i.audit(lambda r: r.by("bo"))).update().meta.stamp.updated == "2026"
assert doc.meta.audit.by == "bo" and doc.meta.audit.updated == 1  # starts from the value already set
assert B.Doc().meta(doc.meta).create().meta.audit.by == "bo"  # a record is a value
with raises(AttributeError, match="intersection values are read-only; use a builder"):
    doc.code.low = 2
with raises(AttributeError, match="'mid' is not a part of the intersection"):
    B.Doc().code(lambda i: i.mid(5))
with raises(TypeError, match="property 'code' takes an intersection value or a Spec, got int"):
    B.Doc().code(7)
B.Doc(doc).code(lambda i: i.clear("high")).update()
assert doc.code.high is None
probed = []
B.Doc(doc).property("meta", lambda p: p.value(lambda a: a.as_intersection(lambda i: (
    i.properties(lambda q: probed.append(q.name())), probed.append(i.has("audit"))))))
assert probed == ["stamp", "audit", True]
with raises(TypeError, match="property 'title' does not hold an intersection"):
    B.Doc().property("title", lambda p: p.value(lambda a: a.as_intersection(lambda i: None)))

# Snapshots: one mapping per part.
B.Doc(doc).code(lambda i: i.high(9)).update()
doc_json = JSON.ToJSON(store)(Doc, doc)
assert doc_json == ('{"root": "s0", "objects": {"s0": {"title": "T", "meta": {"stamp": {"updated": "2026"}, '
                    '"audit": {"by": "bo", "updated": 1}}, "code": {"low": 1, "high": 9}}}}')
for encode, decode in [(Plain.ToPlain(store), Plain.FromPlain), (JSON.ToJSON(store), JSON.FromJSON), (YAML.ToYAML(store), YAML.FromYAML)]:
    assert JSON.ToJSON(store)(Doc, decode(B)(Doc, encode(Doc, doc))) == doc_json
assert FromPlain(Doc, {"root": "s0", "objects": {"s0": {"code": {"low": 3}}}}).code.low == 3  # decoding allows a missing part
for s0, message in [
    ({"meta": {"x": {}}}, "$.objects.s0.meta.x: the intersection has no part 'x'"),
    ({"meta": {"stamp": {"by": "a"}}}, "$.objects.s0.meta.stamp.by: the value object has no property 'by'"),
    ({"code": {"low": "x"}}, "$.objects.s0.code.low: expected int, got str"),
]:
    with raises(DecodeError, match=message):
        FromPlain(Doc, {"root": "s0", "objects": {"s0": s0}})
with raises(TypeError, match="property 'name' is not an intersection"):
    Plain.ToPlain(store)(Contact, Written("Contact", lambda v: v.property("name", lambda p: p.value(
        lambda a: a.as_intersection(lambda i: None)))))


def code(write):
    return Written("Doc", lambda v: v.property("code", lambda p: p.value(lambda a: a.as_intersection(write))))


# Validation: every part present, each of its type.
assert validate(Doc, doc) == []
assert validate(Doc, Fake("Doc", {"code": 1})) == ["Doc#0.code: expected an intersection value, got int"]
assert validate(Doc, code(lambda i: set_property(i, "low", 1))) == ["Doc#0.code: parts ['high'] are not set"]
assert validate(Doc, code(lambda i: (set_property(i, "low", "x"), set_property(i, "high", 2), set_property(i, "mid", 3)))) == [
    "Doc#0.code.low: expected int, got str", "Doc#0.code.mid: not a part of the intersection"]
titled = Written("Doc", lambda v: v.property("title", lambda p: p.value(lambda a: a.as_intersection(lambda i: None))))
assert validate(Doc, titled) == ["Doc#0.title: expected str, got an intersection value"]
meta = Validators.properties_of(Validators.properties_of(doc)["meta"])
assert Validators.properties_of(meta["audit"]) == {"by": "bo", "updated": 1}
Folder = S.OfObject.Builder().name("Folder").ref().properties(lambda p: p.name("doc").of(
    S.OfObject.Builder().properties(lambda q: q.name("code").of(Code)).create())).create()
store.register(Folder)
folder = B.Folder().doc(lambda r: r.code(lambda i: i.low(1).high(2))).create()
held = Validators.properties_of(Validators.properties_of(folder)["doc"])  # replays the intersection value it holds
assert Validators.properties_of(held["code"]) == {"low": 1, "high": 2}

# An intersection of unions: each part holds one branch.
Ids = S.OfIntersection.Builder().parts(lambda p: p.name("mine").of(Id), lambda p: p.name("theirs").of(Id)).create()
Pairing = S.OfObject.Builder().name("Pairing").ref().properties(lambda p: p.name("ids").of(Ids)).create()
store.register(Pairing)
pairing = B.Pairing().ids(lambda i: i.mine(lambda u: u.number(1)).theirs(lambda u: u.code("t"))).create()
assert pairing.ids.mine.number == 1 and pairing.ids.theirs.code == "t" and validate(Pairing, pairing) == []
assert JSON.ToJSON(store)(Pairing, pairing) == '{"root": "s0", "objects": {"s0": {"ids": {"mine": {"number": 1}, "theirs": {"code": "t"}}}}}'
assert JSON.ToJSON(store)(Pairing, FromPlain(Pairing, Plain.ToPlain(store)(Pairing, pairing))) == JSON.ToJSON(store)(Pairing, pairing)

# Comparison: part by part.
assert C.OfObject(Doc, doc).compare(C.OfObject(Doc, B.Doc(doc).clone())) == 0
assert C.OfObject(Doc, doc).compare(C.OfObject(Doc, B.Doc(doc).meta(lambda i: i.audit(lambda r: r.by("x"))).clone())) is None
parts = C.OfIntersection(Code)
assert parts.compare(C.OfIntersection(Code)) == 0  # both are absent
set_property(parts, "low", 1)
assert parts.compare(C.OfIntersection(Code)) is None and C.OfIntersection(Code).compare(parts) is None
assert parts.has("low") and not parts.has("high") and parts.properties(lambda p: None) is parts
low = C.OfIntersection(Code)
set_property(low, "low", 1)
assert parts.compare(low) == 0
Code2 = S.OfIntersection.Builder().parts(lambda p: p.name("low").of(lambda t: t.as_native(int)),
                                         lambda p: p.name("high").of(lambda t: t.as_native(int))).create()
assert parts.compare(C.OfIntersection(Code2)) is None  # another intersection
set_property(parts, "high", 2)
assert parts.clear("high").has("high") is False
kept = C.OfAny(Code).as_intersection(lambda i: set_property(i, "low", 2))
kept.as_intersection(lambda i: set_property(i, "low", 1))  # writing again reuses it
assert kept.compare(C.OfAny(Code).as_intersection(lambda i: set_property(i, "low", 1))) == 0
assert kept.compare(C.OfAny(S.OfNative.Data(int)).as_native(lambda n: n.set(1))) is None

## EMB-09 · Value objects have identities and adjacencies, and belong to one owner
A value object keeps its identity through edits of its owner; set into another owner, or cloned with its owner, it is
copied with its entries, and entries among the copied objects link the copies; replaced or cleared, it is removed with
every entry linking it. Snapshots write a linked value object with its symbol, `$id`, and refer to it without a schema.

In [ ]:
Connection = S.OfRelation.Builder().name("Connection").links("net", "port").create()
Peering = S.OfRelation.Builder().name("Peering").links("a", "b").create()
Port = S.OfObject.Builder().name("Port").properties(text("pin")).relations(
    lambda r: r.name("nets").of(Connection).me("port"), lambda r: r.name("peers").of(Peering).me("a"),
    lambda r: r.name("partners").of(Peering).me("b")).create()
Socket = S.OfUnion.Builder().branches(lambda b: b.name("jack").of(Port), lambda b: b.name("plug").of(Port)).create()
Duo = S.OfIntersection.Builder().parts(lambda p: p.name("left").of(Port), lambda p: p.name("right").of(Port)).create()
Board = S.OfObject.Builder().name("Board").ref().properties(
    text("name"), lambda p: p.name("power").of(Port), lambda p: p.name("ground").of(Port),
    lambda p: p.name("slot").of(Socket), lambda p: p.name("pair").of(Duo)).create()
Net = S.OfObject.Builder().name("Net").ref().properties(text("name"), lambda p: p.name("probe").of(Port)).relations(
    lambda r: r.name("ports").of(Connection).me("net")).create()
for schema in [Board, Net, Connection, Peering]:
    store.register(schema)
assert Port.validate() == [] and Board.validate() == []


def linked(value, adjacency):
    """The objects that `value`'s entries in `adjacency` link, in order."""
    record = Validators._ObjectRecord()
    value.accept(record)
    return [t for entry in record.adjacency_entries.get(adjacency, []) for t in entry.targets.values()]


# Identity and ownership.
cpu = B.Board().name("cpu").power(lambda p: p.pin("1")).ground(lambda p: p.pin("2")).create()
power, ground = cpu.power, cpu.ground
assert power.owner() is cpu and cpu.owner() is None and power.identity() != ground.identity()
assert power.schema_name() == "" and B.Board(cpu).name("CPU").update().power is power  # an update keeps it
vcc = B.Net().name("vcc").ports(lambda x: x.port(power)).create()
B.Board(cpu).power(lambda p: p.pin("1a").peers(lambda x: x.b(ground))).update()  # an edit keeps the identity
assert cpu.power is power and power.pin == "1a"
assert linked(vcc, "ports") == [power] and linked(power, "nets") == [vcc]
assert linked(power, "peers") == [ground] and linked(ground, "partners") == [power]

# Copies: into another owner, and with a clone of the owner; entries among the copies link the copies.
gpu = B.Board().name("gpu").power(power).ground(ground).create()
assert gpu.power is not power and gpu.power.pin == "1a" and gpu.power.owner() is gpu
assert linked(gpu.power, "nets") == [vcc] and linked(gpu.power, "peers") == [gpu.ground]
assert {id(t) for t in linked(vcc, "ports")} == {id(power), id(gpu.power)}
twin = B.Board(cpu).name("twin").clone()
assert twin.power is not power and linked(twin.power, "peers") == [twin.ground] and linked(cpu.power, "peers") == [ground]
edited = B.Board(cpu).power(lambda p: p.pin("1b")).clone()  # an edit in a clone is a copy too
assert edited.power is not power and edited.power.pin == "1b" and power.pin == "1a"
assert linked(edited.power, "peers") == [edited.ground] and linked(edited.ground, "partners") == [edited.power]

# Removal: a value object replaced or cleared goes, with every entry linking it.
B.Board(twin).clear("power").update()
assert {id(t) for t in linked(vcc, "ports")} == {id(power), id(gpu.power), id(edited.power)}
assert linked(twin.ground, "partners") == []
B.Board(gpu).power(lambda p: p.pin("9")).update()
B.Board(gpu).power(B.Board().power(lambda p: p.pin("new")).create().power).update()  # replaced by a copy
assert gpu.power.pin == "new" and linked(gpu.power, "nets") == [] and linked(gpu.ground, "partners") == []
seen = []
B.Board(cpu).property("power", lambda p: p.value(lambda a: a.as_object(lambda r: r.adjacencies(
    lambda adj: seen.append(adj.name())))))
assert seen == ["nets", "peers", "partners"]
twice = B.Board().power(lambda p: p.nets(lambda x: x.net(vcc))).power(lambda p: p.pin("p")).create()
assert twice.power.pin == "p" and linked(twice.power, "nets") == [vcc]  # the second edit starts from the first
B.Board(twice).clear("power").update()
builder = B.Net()
assert builder.identify(power) is builder
with raises(TypeError, match="proxies can only link proxy instances and their value objects"):
    B.Net().ports(lambda x: x.port(object())).create()
with raises(AttributeError, match="'wires' is not an adjacency of the value object"):
    B.Board().power(lambda p: p.adjacency("wires", lambda a: None))
with raises(TypeError, match="'nets' takes an entry Spec"):
    B.Board().power(lambda p: p.nets(vcc))
store.register(Port)
assert power.schema_name() == "Port"  # a value object's schema may be registered too

# Snapshots: the linked value object carries its symbol, and references to it carry none.
board_json = JSON.ToJSON(store).Reachable(Board, cpu)
assert board_json == (
    '{"root": "s0", "objects": {"s0": {"name": "CPU", "power": {"$id": "s3", "pin": "1a", "nets": [{"net": {"$ref": '
    '"s1", "$schema": "Net"}}], "peers": [{"b": {"$ref": "s4"}}]}, "ground": {"$id": "s4", "pin": "2", "partners": '
    '[{"a": {"$ref": "s3"}}]}}, "s1": {"name": "vcc", "ports": [{"port": {"$ref": "s3"}}, {"port": {"$ref": "s5"}}]}, '
    '"s2": {"$schema": "Board", "name": "CPU", "power": {"$id": "s5", "pin": "1b", "nets": [{"net": {"$ref": "s1", '
    '"$schema": "Net"}}], "peers": [{"b": {"$ref": "s6"}}]}, "ground": {"$id": "s6", "pin": "2", "partners": [{"a": '
    '{"$ref": "s5"}}]}}}}')
assert JSON.ToJSON(store)(Board, gpu) == '{"root": "s0", "objects": {"s0": {"name": "gpu", "power": {"pin": "new"}, "ground": {"pin": "2"}}}}'
for encode, decode in [(Plain.ToPlain(store), Plain.FromPlain), (JSON.ToJSON(store), JSON.FromJSON), (YAML.ToYAML(store), YAML.FromYAML)]:
    back = decode(B).Reachable(Board, encode.Reachable(Board, cpu))
    assert JSON.ToJSON(store).Reachable(Board, back) == board_json
    assert back.power.owner() is back and linked(back.power, "peers") == [back.ground]
    assert linked(back.power, "nets")[0].name == "vcc" and linked(linked(back.power, "nets")[0], "ports")[0] is back.power

snapshot = Plain.ToPlain(store).Reachable(Board, cpu)
for change, message in [
    (lambda s: s["objects"]["s0"]["ground"].__setitem__("$id", "s3"), "$.objects.s0.ground.$id: 's3' is the symbol of two objects"),
    (lambda s: s["objects"]["s0"]["ground"].__setitem__("$id", "s1"), "$.objects.s0.ground.$id: 's1' is the symbol of two objects"),
    (lambda s: s["objects"]["s0"]["ground"].__setitem__("$id", 4), "$.objects.s0.ground.$id: a symbol must be a string, got int"),
    (lambda s: s["objects"]["s0"]["power"]["peers"][0].__setitem__("b", {"$ref": "s9"}),
     "$.objects.s0.power.peers[0].b: unresolved reference 's9': no value object in the snapshot has this symbol"),
    (lambda s: s["objects"]["s0"]["power"]["peers"][0].__setitem__("b", {"$ref": "s1"}),
     "$.objects.s0.power.peers[0].b: unresolved reference 's1': no value object in the snapshot has this symbol"),
    (lambda s: s["objects"]["s0"]["power"].__setitem__("nets", "x"), "$.objects.s0.power.nets: an adjacency must be a list of entries"),
    (lambda s: s["objects"]["s2"].__setitem__("$schema", 1), "$.objects.s2.$schema: a schema name must be a string, got int"),
    (lambda s: s["objects"]["s2"].__setitem__("$schema", "Net"), "$.objects.s2.power: 'Net' has no property or adjacency 'power'"),
    (lambda s: s["objects"]["s2"].pop("$schema"), "$.objects.s2: nothing references this object, so its schema is unknown"),
    (lambda s: s["objects"]["s1"].__setitem__("$schema", "Board"),
     "$.objects.s1.$schema: 's1' is referenced as both 'Net' and 'Board'"),
]:
    broken = copy.deepcopy(snapshot)
    change(broken)
    with raises(DecodeError, match=message):
        Plain.FromPlain(B).Reachable(Board, broken)

# Reachability goes through value objects to their owners, and validation checks value objects' entries.
assert Reachable.of(vcc) == [vcc, cpu, edited]
assert Validators.Validate(B).Reachable(Net, vcc) == []
odd = B.Net().name("odd").ports(lambda x: x.port(eve.reach.phone)).create()  # a phone cannot fill 'port'
assert Validators.Validate(B)(Net, odd) == []  # the phone's owner is not validated, so its schema is not known
assert Validators.Validate(B).Reachable(Net, odd) == [
    "Net#0.ports[0].port: a value object cannot fill link 'port'; its schema declares no adjacency to this relation via 'port'"]
crossed = B.Board().power(lambda p: p.peers(lambda x: x.b(power))).ground(lambda p: p.nets(lambda x: x.net(vcc))).create()
assert Validators.Validate(B)(Board, crossed) == []
stray = Written("Board", lambda v: v.property("power", lambda p: p.value(lambda a: a.as_object(lambda o: (
    o.identify(power), o.adjacency("wires", lambda w: None), o.adjacency("nets", lambda w: w.add(lambda e: e.link(
        "net", lambda k: k.set(eve)))))))))
anonymous = Written("Board", lambda v: v.property("power", lambda p: p.value(lambda a: a.as_object(lambda o: o.adjacency(
    "nets", lambda w: w.add(lambda e: e.link("net", lambda k: k.set(vcc))))))))
assert Validators.Validate(B)(Board, anonymous) == []  # a value object that does not identify itself is not linked
assert Validators.Validate(B)(Board, stray) == [
    "Board#0.power.wires: not an adjacency of the value object",
    "Board#0.power.nets[0].net: a 'Contact' cannot fill link 'net'; its schema declares no adjacency to this relation via 'net'"]

# Value objects inside union and intersection values, and a reference object's entries to its own value objects.
rack = B.Board().name("rack").slot(lambda u: u.jack(lambda p: p.pin("j"))).pair(
    lambda i: i.left(lambda p: p.pin("l")).right(lambda p: p.pin("r"))).create()
bus = B.Net().name("bus").ports(lambda x: x.port(rack.slot.jack)).ports(lambda x: x.port(rack.pair.left)).create()
bus_json = JSON.ToJSON(store).Reachable(Net, bus)
assert '"s1": {"$schema": "Board", "name": "rack"' in bus_json  # linked only through its value objects
back = Plain.FromPlain(B).Reachable(Net, Plain.ToPlain(store).Reachable(Net, bus))
assert JSON.ToJSON(store).Reachable(Net, back) == bus_json
assert [port.pin for port in linked(back, "ports")] == ["j", "l"] and linked(back, "ports")[0].owner().owner().name == "rack"
probe = B.Net().name("probe").probe(lambda p: p.pin("t")).create()
B.Net(probe).ports(lambda x: x.port(probe.probe)).update()
twin_probe = B.Net(probe).clone()
assert linked(twin_probe, "ports") == [twin_probe.probe] and linked(probe, "ports") == [probe.probe]

# Comparison: value objects compare deeply, their entries included; links among them compare by where they are.
same_board = C.OfObject(Board, cpu).compare
assert same_board(C.OfObject(Board, B.Board(cpu).clone())) == 0  # copies, linked among themselves the same way
bare = B.Board(cpu).power(B.Board().power(lambda p: p.pin("1a")).create().power).clone()
assert same_board(C.OfObject(Board, bare)) is None  # the same pins, but the power's entries differ
elsewhere = B.Board().name("CPU").power(lambda p: p.pin("1a").peers(lambda x: x.b(gpu.ground)).nets(lambda x: x.net(vcc))
                                         ).ground(lambda p: p.pin("2")).create()
assert same_board(C.OfObject(Board, elsewhere)) is None  # its power links another board's ground
assert C.OfObject(Net, vcc).compare(C.OfObject(Net, B.Net(vcc).clone())) == 0  # a reference object's own entries do not count

## EMB-10 · Entry properties may be value objects
An entry property's value object belongs to the entry, is written nested in it, and compares, and keys the entry, by
its properties. It cannot have adjacencies: an entry is written under each object it links.

In [ ]:
Span = S.OfObject.Builder().properties(text("start"), text("end")).create()
Booking = S.OfRelation.Builder().name("Booking").links("guest", "room").properties(lambda p: p.name("stay").of(Span), text("note")).unique(
    "note").create()
Guest = S.OfObject.Builder().name("Guest").ref().properties(text("name")).relations(lambda r: r.name("rooms").of(Booking).me("guest")).create()
Room = S.OfObject.Builder().name("Room").ref().properties(text("number")).relations(lambda r: r.name("guests").of(Booking).me("room")).create()
for schema in [Guest, Room, Booking]:
    store.register(schema)
assert Booking.validate() == []
Ported = S.OfObject.Builder().properties(text("pin")).relations(
    lambda r: r.name("nets").of(S.OfRelation.Builder().links("net", "port").create()).me("port")).create()
assert S.OfRelation.Builder().links("a", "b").properties(lambda p: p.name("port").of(Ported)).create().validate() == [
    "property 'port': a value object held by an entry cannot have adjacencies"]


def rows(value, adjacency):
    """The recorded entries of `value` in `adjacency`."""
    record = Validators._ObjectRecord()
    value.accept(record)
    return record.adjacency_entries.get(adjacency, [])


room = B.Room().number("101").create()
guest = B.Guest().name("Ann").rooms(lambda x: x.room(room).stay(lambda s: s.start("mon").end("wed"))).create()
B.Guest(guest).rooms(lambda x: x.room(room).stay(lambda s: s.start("mon").end("wed"))).update()
assert len(rows(guest, "rooms")) == 1  # an equal entry is elided: its value object compares by its properties
stay = rows(guest, "rooms")[0].values["stay"]
assert Validators.properties_of(stay) == {"start": "mon", "end": "wed"} and stay.target.owner() is guest
assert len(rows(room, "guests")) == 1 and Validators.Validate(B)(Guest, guest) == []

guest_json = JSON.ToJSON(store).Reachable(Guest, guest)
assert guest_json == ('{"root": "s0", "objects": {"s0": {"name": "Ann", "rooms": [{"room": {"$ref": "s1", "$schema": "Room"}, '
                      '"stay": {"start": "mon", "end": "wed"}}]}, "s1": {"number": "101", "guests": [{"guest": {"$ref": '
                      '"s0", "$schema": "Guest"}, "stay": {"start": "mon", "end": "wed"}}]}}}')
assert JSON.ToJSON(store).Reachable(Guest, JSON.FromJSON(B).Reachable(Guest, guest_json)) == guest_json
broken = copy.deepcopy(Plain.ToPlain(store).Reachable(Guest, guest))
broken["objects"]["s0"]["rooms"][0]["stay"]["$id"] = "s9"
with raises(DecodeError, match="$.objects.s0.rooms[0].stay.$id: the value object has no property '$id'"):
    Plain.FromPlain(B).Reachable(Guest, broken)

B.Guest(guest).rooms(lambda x: x.room(room).stay(lambda s: s.start("mon").end("wed")).note("late")).update()
assert Validators.Validate(B)(Guest, guest) == [
    "unique(note) violated: entries agreeing on ['guest', 'room', 'stay'] differ on ['note']"]

## EMB-11 · Flat unions and intersections: a union value read as its branch's, set by type; parts' properties as the value's own

In [ ]:
Mail = S.OfObject.Builder().name("Mail").properties(text("address")).create()
Call = S.OfObject.Builder().name("Call").properties(text("number")).create()
Channel = S.OfUnion.Builder().name("Channel").branches(lambda b: b.name("call").of(Call), lambda b: b.name("mail").of(Mail)).flat().create()
Code = S.OfUnion.Builder().name("Code").branches(lambda b: b.name("n").of(lambda t: t.as_native(int)),
                                                 lambda b: b.name("s").of(lambda t: t.as_native(str))).flat().create()
When = S.OfObject.Builder().name("When").properties(text("at", int)).create()
Who = S.OfObject.Builder().name("Who").properties(text("by")).create()
Stamped = S.OfIntersection.Builder().name("Stamped").parts(lambda b: b.name("when").of(When), lambda b: b.name("who").of(Who)).flat().create()
Ticket = S.OfObject.Builder().name("Ticket").ref().properties(
    lambda p: p.name("channel").of(Channel), lambda p: p.name("code").of(Code),
    lambda p: p.name("codes").of(lambda t: t.as_indexed(lambda i: i.of(Code))),
    lambda p: p.name("by_name").of(lambda t: t.as_indexed(lambda i: i.key(lambda k: k.as_native(str)).of(Code))),
    lambda p: p.name("stamp").of(Stamped)).create()
assert [s.validate() for s in (Channel, Code, Stamped, Ticket)] == [[], [], [], []] and Channel.flat and not Reach.flat
flats = Proxies.OfStore()
flats.register(Ticket)
ticket = flats.Ticket().channel(lambda u: u.mail(lambda m: m.address("a@x"))).code(7).codes([1, "x"]).by_name(
    {"a": 2}).stamp(lambda s: s.at(3).by("me")).create()
assert (ticket.channel.address, ticket.code, ticket.codes, ticket.by_name["a"]) == ("a@x", 7, (1, "x"), 2)  # the branches' values
assert (ticket.stamp.at, ticket.stamp.by, ticket.stamp.when.at) == (3, "me", 3)  # the parts' properties, as its own
other = flats.Ticket().code("z").channel(ticket.channel).stamp(lambda s: s.by("you")).create()  # set by type
assert (other.code, other.channel.address, other.stamp.by, other.stamp.at) == ("z", "a@x", "you", None)
text_ = JSON.ToJSON(flats).Reachable(Ticket, ticket)
assert '"channel": {"mail": {"address": "a@x"}}, "code": {"n": 7}, "codes": [{"n": 1}, {"s": "x"}]' in text_  # tagged
assert JSON.FromJSON(flats).Reachable(Ticket, text_).code == 7
with raises(TypeError, match="property 'code': a float is not a value of any branch of the flat union"):
    flats.Ticket().code(1.5)
with raises(AttributeError, match="'nope' is not a part of the intersection"):
    flats.Ticket().stamp(lambda s: s.nope(1))
# Validation: a flat union's branches told apart by type; a flat intersection's parts objects, their properties distinct.
twice = S.OfUnion.Builder().branches(lambda b: b.name("a").of(lambda t: t.as_native(str)),
                                     lambda b: b.name("b").of(lambda t: t.as_native(str))).flat().create()
lists = S.OfUnion.Builder().branches(lambda b: b.name("a").of(lambda t: t.as_indexed(lambda i: i.of(Call))),
                                     lambda b: b.name("b").of(lambda t: t.as_indexed(lambda i: i.of(Mail)))).flat().create()
maps = S.OfUnion.Builder().branches(lambda b: b.name("a").of(lambda t: t.as_indexed(lambda i: i.key(Call).of(Call))),
                                    lambda b: b.name("b").of(lambda t: t.as_indexed(lambda i: i.key(Mail).of(Mail)))).flat().create()
same = S.OfUnion.Builder().branches(lambda b: b.name("a").of(Call), lambda b: b.name("b").of(Call)).flat().create()
foreign = S.OfUnion.Builder().branches(lambda b: b.name("a").of(lambda t: t.as_native(lambda n: n.token("ccpp", "x"))),
                                       lambda b: b.name("b").of(lambda t: t.as_native(lambda n: n.token("ccpp", "x")))).flat().create()
assert [u.validate() for u in (twice, lists, maps, same, foreign)] == [
    ["flat union: branches 'a' and 'b' are not told apart by type"]] * 5
clash = S.OfIntersection.Builder().parts(lambda b: b.name("a").of(When), lambda b: b.name("b").of(When)).flat().create()
natives = S.OfIntersection.Builder().parts(lambda b: b.name("a").of(lambda t: t.as_native(int)),
                                           lambda b: b.name("b").of(lambda t: t.as_native(str))).flat().create()
assert clash.validate() == ["flat intersection: parts 'a' and 'b' both declare 'at'"]
assert natives.validate() == ["flat intersection: part 'a' is not an object schema",
                              "flat intersection: part 'b' is not an object schema"]
assert flats.Ticket().stamp(lambda s: s.at(1)).create().stamp.by is None  # a part not set
Bag = S.OfUnion.Builder().name("Bag").branches(
    lambda b: b.name("items").of(lambda t: t.as_indexed(lambda i: i.of(lambda x: x.as_native(int)))),
    lambda b: b.name("named").of(lambda t: t.as_indexed(lambda i: i.key(lambda k: k.as_native(str)).of(
        lambda x: x.as_native(int))))).flat().create()
flats.register(S.OfObject.Builder().name("Holder").ref().properties(lambda p: p.name("bag").of(Bag)).create())
assert flats.Holder().bag([1, 2]).create().bag == (1, 2) and flats.Holder().bag({"a": 1}).create().bag["a"] == 1
modules = Proxies.OfStore()
read = Modules.schemas(modules, Modules.module(modules, [Channel, Stamped, When, Who, Call, Mail]))
assert read["Channel"].flat and read["Stamped"].flat and '"flat": true' in JSON.ToJSON(modules)(
    S.Module.Schema, Modules.module(modules, [Channel]))